In [1]:
import pandas as pd

# 1. Cargamos el punto de control cronológicamente
print("Cargando datos limpios...")
df = pd.read_csv('../data/interim/partidos_limpios.csv')

# Aseguramos el orden temporal estricto (Crucial para no hacer "trampa" matemática)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# 2. Funciones Matemáticas Puras del Sistema Elo
def calcular_resultado_esperado(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def actualizar_elo(rating_actual, resultado_esperado, resultado_real, k_factor=30):
    return rating_actual + k_factor * (resultado_real - resultado_esperado)

# 3. Inicializamos la "Memoria"
elo_equipos = {}
elo_inicial = 1500
elo_local_historico = []
elo_visitante_historico = []

print("Iniciando motor histórico Elo...")

# 4. La Máquina del Tiempo (Bucle Cronológico)
for index, row in df.iterrows():
    local = row['home_team']
    visitante = row['away_team']

    # Asignar 1500 si es el primer partido histórico de esa selección
    if local not in elo_equipos: elo_equipos[local] = elo_inicial
    if visitante not in elo_equipos: elo_equipos[visitante] = elo_inicial

    # Guardar el nivel PREVIO al partido (Lo que el modelo XGBoost usará para predecir)
    elo_local_historico.append(elo_equipos[local])
    elo_visitante_historico.append(elo_equipos[visitante])

    # Convertir target_numeric a matemáticas puras (Victoria=1, Empate=0.5, Derrota=0)
    if row['target_numeric'] == 2:   # Local gana
        res_local, res_visit = 1, 0
    elif row['target_numeric'] == 1: # Empate
        res_local, res_visit = 0.5, 0.5
    else:                            # Visitante gana
        res_local, res_visit = 0, 1

    # Calcular probabilidades de ese partido específico
    prob_local = calcular_resultado_esperado(elo_equipos[local], elo_equipos[visitante])
    prob_visitante = calcular_resultado_esperado(elo_equipos[visitante], elo_equipos[local])

    # Actualizar la memoria con el resultado post-partido
    elo_equipos[local] = actualizar_elo(elo_equipos[local], prob_local, res_local)
    elo_equipos[visitante] = actualizar_elo(elo_equipos[visitante], prob_visitante, res_visit)

# 5. Inyectar las nuevas variables de fuerza (Features) al DataFrame
df['elo_home'] = elo_local_historico
df['elo_away'] = elo_visitante_historico

print("¡Ingeniería de Características finalizada!")
df[['date', 'home_team', 'away_team', 'elo_home', 'elo_away', 'target']].tail()

Cargando datos limpios...
Iniciando motor histórico Elo...
¡Ingeniería de Características finalizada!


,date,home_team,away_team,elo_home,elo_away,target
25098,2026-03-31,Botswana,Malawi,1409.819336,1449.874123,HW
25099,2026-03-31,Mexico,Belgium,1820.608162,1811.547468,D
25100,2026-03-31,Montenegro,Slovenia,1489.902076,1673.626421,AW
25101,2026-03-31,Niger,Togo,1462.310281,1460.241494,AW
25102,2026-03-31,Kazakhstan,Comoros,1435.571896,1476.982249,HW


In [2]:
import os

# 1. Verificamos el Top 10 actual para comprobar que la matemática tiene sentido
ranking_actual = pd.DataFrame(list(elo_equipos.items()), columns=['Equipo', 'Puntuacion_Elo'])
ranking_actual = ranking_actual.sort_values('Puntuacion_Elo', ascending=False).reset_index(drop=True)

print("--- Top 10 Ranking Elo Mundial ---")
print(ranking_actual.head(10).to_string(index=False))

# 2. Guardamos nuestro dataset maestro ya enriquecido
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/features_finales.csv', index=False)
print("\n¡Punto de control final! Dataset listo para el modelo guardado en 'data/processed/features_finales.csv'")

--- Top 10 Ranking Elo Mundial ---
   Equipo  Puntuacion_Elo
    Spain     2005.888333
Argentina     2004.407848
   France     1962.057860
   Brazil     1897.187895
  Morocco     1895.640448
 Portugal     1893.534080
  England     1889.218671
    Japan     1887.409234
 Colombia     1882.355736
  Germany     1882.083065

¡Punto de control final! Dataset listo para el modelo guardado en 'data/processed/features_finales.csv'
